In [1]:
import pandas as pd
import numpy as np
import os
from openai import OpenAI

In [2]:
from dotenv import load_dotenv
load_dotenv(os.path.expanduser("~/final_project_openrouter/.env"))

True

In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [4]:
#Loading in AdminIT dataset -- # Dataset gives us column: [0] = original, [2] = L2 simplified 
adminIt = pd.read_table('/home/c23068554/final_project_openrouter/datasets/adminit/admin-it-l2.txt', header=None, sep='\t', names = ['original', 'L1_simplified', 'L2_simplified'])
#print(original.head(10))

In [5]:
# data splitting, splitting data values to be tested and some to be used as few-shot
from sklearn.model_selection import train_test_split

random_state = 59
data_test, data_train = train_test_split(adminIt, test_size = 0.02, random_state = random_state)

print(len(data_test))
print(len(data_train))

131
3


In [ ]:
#data_test = data_test.head(10)
#print(f"TEST MODE: running on {len(data_test)} sentences only")

# Making prompt with randomized n-shot prompting



In [7]:
instruction = "Si prega di riscrivere la seguente frase complessa per renderla più comprensibile a chi non è madrelingua italiana. È possibile farlo sostituendo le parole complesse con sinonimi più semplici (parafrasando), eliminando le informazioni non importanti (comprimendo) e/o suddividendo la frase complessa in diverse frasi più semplici. La frase semplificata finale deve essere grammaticalmente corretta, scorrevole e conservare le idee principali dell'originale senza alterarne il significato.\n\n"
def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Complesso: {row.loc['original']}\nSemplice:{row.loc['L2_simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, data_train)
print(fewshot_example)

Si prega di riscrivere la seguente frase complessa per renderla più comprensibile a chi non è madrelingua italiana. È possibile farlo sostituendo le parole complesse con sinonimi più semplici (parafrasando), eliminando le informazioni non importanti (comprimendo) e/o suddividendo la frase complessa in diverse frasi più semplici. La frase semplificata finale deve essere grammaticalmente corretta, scorrevole e conservare le idee principali dell'originale senza alterarne il significato.

Complesso: È un contributo economico a sostegno della maternità per le donne che non ricevono altri trattamenti previdenziali di maternità oppure li percepiscono ma sono di importo inferiore (astensione obbligatoria di maternità erogata dall'INPS o altro ente previdenziale).
Semplice:È un contributo economico a sostegno delle donne in maternità che non ricevono altri contributi, o che li ricevono ma con un importo inferiore (come ad esempio il congedo di maternità dell'INPS).

Complesso: 3) assenza di tit

# Load in model and inference

### Decoder models

In [8]:
# Decoder models
mistral_7b = "mistralai/mistral-7b-instruct-v0.1"
llama31_8b = "meta-llama/llama-3.1-8b-instruct"
gemma2_9b = "google/gemma-2-9b-it"
qwen2_7b = "qwen/qwen-2.5-7b-instruct"
llama33_70b = "meta-llama/llama-3.3-70b-instruct"

In [9]:
import time
model_id = llama31_8b
def generate(prompt, max_tokens=200):
    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": "Rispondere solo con il testo semplificato."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=1.0,
                top_p=0.9,
            )
            content = response.choices[0].message.content
            if content is None:
                print(f"  Empty response (attempt {attempt+1})")
                time.sleep(1)
                continue
            return content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Max retries exceeded")

In [10]:
#looping through all n-shot prompt
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  for i, row in enumerate(data_test['original']):
    full_prompt = fewshot_example + f"Complesso: {row}\nSemplice:"
    LMsimplified.append(generate(full_prompt))
    #progress check on infference
    if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(data_test)} done")
  return LMsimplified

### Inference

In [11]:
LMoutput = (promptLoop(fewshot_example, data_test))
#print(promptLoop(fewshot_example, data_test))

  10/131 done
  20/131 done
  30/131 done
  40/131 done
  50/131 done
  60/131 done
  70/131 done
  80/131 done
  90/131 done
  100/131 done
  110/131 done
  120/131 done
  130/131 done


In [12]:
# Clean prompt leakage from decoder outputs, using common LLM speech 
def clean_output(text):
    text = text.lstrip(':.,;>•*- \n')
    for delim in ['\nComplex:', '\n\nComplex:', '\nNote:', '\n(Note:', '\nAnd also', '\ncomplex sentence', 'Complesso', 'Complesso:', 'Semplice', 'Semplice:']:
        if delim in text:
            text = text.split(delim)[0]
    return text.strip()

LMoutput = [clean_output(s) for s in LMoutput]

In [13]:
reference = data_test['L2_simplified'].tolist()
source = data_test['original'].tolist()

# CSV WRITE

In [14]:
import csv

# change to each experiment ID
exp_id = "I-4"
out_path = "lm_output.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, reference, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])
        
print(f"Wrote {len(source)} rows to {out_path}")

Wrote 131 rows to lm_output.csv


# Evaluation

### Preliminary tests on the model
 How many sentences dont actually get simplified by the model?

In [15]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['L2_simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")

Le regioni e le province autonome nelle quali siano presenti aziende sanitarie nelle quali risultino non disponibili gli spazi per l'esercizio dell'attività libero professionale, possono autorizzare, limitatamente alle medesime aziende sanitarie, l'adozione di un programma sperimentale che preveda lo svolgimento delle stesse attività, in via residuale, presso gli studi privati dei professionisti collegati in rete, ai sensi di quanto previsto dalla lettera a-bis) del presente comma, previa sottoscrizione di una convenzione annuale rinnovabile tra il professionista interessato e l'azienda sanitaria di appartenenza, sulla base di uno schema tipo approvato con accordo sancito dalla Conferenza permanente per i rapporti tra lo Stato, le regioni e le province autonome di Trento e di Bolzano.
Le regioni e le province autonome possono autorizzare le aziende sanitarie che non hanno spazi per l'esercizio dell'attività libero professionale a svolgere queste attività presso gli studi privati di alt

### ROUGE, BLEU, BERTScore, SARI

In [16]:
import evaluate

simple = data_test['L2_simplified'].tolist()

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
references = [[s] for s in data_test["L2_simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= references)

# Compute scores
rouge_results = rouge.compute(predictions=LMoutput, references=reference)
bleu_results = bleu.compute(predictions=LMoutput, references=reference)

#in AdminIT paper, xlm-roberta-large is used
bertscore_compute = bertscore.compute(predictions=LMoutput, references=reference, lang='it', model_type='xlm-roberta-large')
berstcoreAvg = np.mean(bertscore_compute['f1'])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### FKGL Score

In [17]:
import textstat
textstat.set_lang("it")

originalAvg = np.mean([textstat.flesch_reading_ease(s) for s in data_test["original"].tolist()])
simplifiedAvg = np.mean([textstat.flesch_reading_ease(s) for s in LMoutput])

### Gulpease Index

In [18]:
gulOrig = np.mean([textstat.gulpease_index(s) for s in data_test["original"].tolist()])
gulSimp = np.mean([textstat.gulpease_index(s) for s in LMoutput])

In [19]:
#output
print(f"Dataset: Admin-IT-L2, size: {len(data_test)}, random state: {random_state}")
print(f"Model: {model_id}")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score (xlm-roberta-large): {berstcoreAvg}")
print(f"Sari Score: {sari_score['sari']}")
print()
print(f"Original Flesch score: {originalAvg}")
print(f"Simplified Flesch score: {simplifiedAvg}")
print()
print(f"Original Gulpease Index: {gulOrig}")
print(f"Simplified Gulpease Index: {gulSimp}")
print(f"unsimplified sentences: {count}/{len(data_test)}")

Dataset: Admin-IT-L2, size: 131, random state: 59
Model: mistralai/mistral-7b-instruct-v0.1

ROUGE Score: 0.2617204569389774
BLEU Score: 0.07784957120179753
BERTScore Score (xlm-roberta-large): 0.9023071481981351
Sari Score: 37.53206800349709

Original Flesch score: 24.414303406546008
Simplified Flesch score: 51.4020593600782

Original Gulpease Index: 45.9101151152805
Simplified Gulpease Index: 48.850180510271244
unsimplified sentences: 0/131


In [20]:
print(f"\nTAB-SEPARATED (paste into experiment sheet metrics columns):")
print(f"{sari_score['sari']:.4f}\t{rouge_results['rouge1']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{gulSimp:.4f}\t{count}/{len(data_test)}")



TAB-SEPARATED (paste into experiment sheet metrics columns):
37.5321	0.2617	0.0778	0.9023	48.8502	0/131


In [21]:
# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

#check disk space used:
# du -h --max-depth=1 ~ | sort -h

'''
Exporting CSV:
On local:
scp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'
On Remote: 
rm lm_output.csv

'''

# remove .env using: rm ~/final_project_openrouter/.env

"\nExporting CSV:\nOn local:\nscp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'\nOn Remote: \nrm lm_output.csv\n\n"